## AuxTel Hexapod Query - 15-Mar-21

In this notebook, investigate hexapod status\
Craig Lage - 15Mar21

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.time import Time, TimeDelta
from lsst_efd_client import EfdClient
from lsst.daf.persistence import Butler

In [2]:
# Get EFD client and the butler
client = EfdClient('summit_efd')
REPO_DIR = '/project/shared/auxTel/rerun/quickLook'
butler = Butler(REPO_DIR)

In [3]:
# Find the time of exposure
expId=2021031100119
tStart = butler.queryMetadata('raw', ['DATE'], detector=0, expId=expId)# Get the data
t_start = Time(tStart, scale='tai')[0]
t_end = t_start + TimeDelta(1, format='sec')

In [ ]:
# Get the reported position
hex_position = await client.select_time_series("lsst.sal.ATHexapod.positionStatus", ['*'],
                                          t_start, t_end)
# This dictionary gives the hexapod names and indices
names = {'u':3,'v':4,'w':5,'x':0,'y':1,'z':2}
positions = {}
for name in names.keys():
    key = 'reportedPosition%d'%names[name]
    position = hex_position[key][0]
    positions[name] = position
    
print(positions)

In [8]:
async def getHexapod(expId):
    # Find the time of exposure
    tStart = butler.queryMetadata('raw', ['DATE'], detector=0, expId=expId)# Get the data
    t_start = Time(tStart, scale='tai')[0]
    t_end = t_start + TimeDelta(1, format='sec')
    # Get the reported position
    hex_position = await client.select_time_series("lsst.sal.ATHexapod.positionStatus", ['*'],
                                              t_start, t_end)
    # This dictionary gives the hexapod names and indices
    names = {'u':3,'v':4,'w':5,'x':0,'y':1,'z':2}
    positions = {}
    for name in names.keys():
        key = 'reportedPosition%d'%names[name]
        position = hex_position[key][0]
        positions[name] = position
    print(expId, positions)
    return positions  

In [ ]:
expId=2021031100119
for i in range(50):
    expId += 1
    await getHexapod(expId)

In [10]:
seqNos = [119, 156, 184, 276, 293]
for seq in seqNos:
    expId = '2021031100' + str(seq)
    pos = await getHexapod(expId)
    print(f"{expId} has Hex z = {pos['z']}")

2021031100119 {'u': 0.349998939549, 'v': 0.219998427887, 'w': -1.21334438207e-06, 'x': -5.22001589681, 'y': 0.776301688756, 'z': 0.159955282556}
2021031100119 has Hex z = 0.159955282556
2021031100156 {'u': 0.350000482433, 'v': 0.219999913918, 'w': 3.32066214675e-07, 'x': -5.08898730559, 'y': 0.889290321634, 'z': 0.164316156895}
2021031100156 has Hex z = 0.164316156895
2021031100184 {'u': 0.350001016276, 'v': 0.219997353204, 'w': -5.92456539785e-07, 'x': -5.08685879631, 'y': 0.884520321717, 'z': 0.215608462407}
2021031100184 has Hex z = 0.215608462407
2021031100276 {'u': 0.349999867363, 'v': 0.219999615414, 'w': -1.30335942438e-06, 'x': -5.11155952972, 'y': 0.770681752184, 'z': 0.29509436778}
2021031100276 has Hex z = 0.29509436778
2021031100293 {'u': 0.350000881839, 'v': 0.219999222855, 'w': -1.16734905761e-06, 'x': -5.11324062217, 'y': 0.773987165678, 'z': 0.236571453485}
2021031100293 has Hex z = 0.236571453485
